In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.iteration import iterate_calibration
from vetuner.metrics import evaluate_afr, evaluate_table
from vetuner.pulse_width import (export_table_csv,pulse_width_error,pulse_width_map,required_injector_flow,)
from vetuner.ve_surface import BASE_MAP_SURFACE, TRUE_SURFACE, ve_table

In [ ]:
"""
Calculate the pulse width for the calibrated engine.
"""


train = generate_drive_cycle()
test = generate_drive_cycle(cycle=VALIDATION_CYCLE)
base, truth = ve_table(params=BASE_MAP_SURFACE), ve_table(params=TRUE_SURFACE)

history = iterate_calibration(train, base, gain=0.6, max_passes=5, method="gaussian_process")
calibrated = history.final_table

for name, table in [("starting map", base), ("calibrated", calibrated)]:
    errors = pulse_width_error(table, truth)
    print(f"{name:14s} pulse width error: {errors['mean_abs_ms']:.3f} ms mean, "
          f"{errors['max_abs_ms']:.3f} ms max, {errors['mean_abs_pct']:.2f}% mean, "
          f"{errors['fraction_within_1pct']:.0%} within 1%")

result = pulse_width_map(calibrated)
print(f"\nmax duty {result.max_duty:.1%} at {result.worst_cell[0]:.0f} rpm, "
      f"{result.worst_cell[1]:.0f} kPa")
print(f"all cells deliverable: {result.feasible.all()}")
print(f"flow needed for 80% duty: {required_injector_flow(calibrated):.0f} cc/min "
      f"(fitted: 440 cc/min)")

export_table_csv("outputs/ve_table_calibrated.csv", calibrated)
export_table_csv("outputs/pulse_width_ms.csv", result.pulse_width_ms,
                 label="PW ms", scale=1.0, decimals=3)
print("\nwritten to outputs/")

In [ ]:
extent = [result.rpm_axis[0], result.rpm_axis[-1], result.map_axis[0], result.map_axis[-1]]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

reference = pulse_width_map(truth)
panels = [
    (result.pulse_width_ms, "Commanded pulse width (ms)", "viridis", None),
    (result.duty * 100, "Injector duty (%)", "plasma", None),
    (result.pulse_width_ms - reference.pulse_width_ms, "Error vs true table (ms)", "RdBu_r", True),
]

for ax, (data, title, cmap, symmetric) in zip(axes, panels, strict=True):
    limit = np.abs(data).max() if symmetric else None
    im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap,
                   vmin=-limit if symmetric else None, vmax=limit if symmetric else None)
    ax.set_title(title)
    ax.set_xlabel("rpm")
    fig.colorbar(im, ax=ax)

axes[0].set_ylabel("MAP (kPa abs)")
fig.suptitle(f"Injector output at {result.iat_c:.0f} °C, {result.batt_v:.1f} V")
fig.tight_layout()